In [0]:
from pyspark.sql import functions as f
from datetime import datetime
from delta.tables import DeltaTable
import uuid

In [0]:
spark.sql("use catalog ecommerce_data")
spark.sql("create schema if not exists silver")

silver_run_id=str(uuid.uuid4())

print("Silver run id:",silver_run_id)

In [0]:
spark.sql("""
          create table if not exists ecommerce_data.silver.silver_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_ingested_at timestamp,
              rows_merged bigint,
              silver_run_id string,
              run_status string,
              updated_at timestamp
          )
          using delta
          """)

1st helper function
upsert_to_silver() -> if table exists update records 
else create new table

In [0]:
def upsert_to_silver(source_table,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        df=DeltaTable.forName(spark,target_table)
        (
            df.alias('target')
            .merge(source_table.alias('source'),f"target.{join_key}=source.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        source_table.write.format('delta').mode('append').saveAsTable(target_table) 


2nd helper function

gets the latest ingested at timestamp processed by silver from bronze

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
        lp=(spark.table("ecommerce_data.silver.silver_control").filter(
                  (f.col('layer')=='silver') &
                  (f.col('entity_name')==entity_name) &
                  (f.col('run_status')=='success'))
              .orderBy(f.col('updated_at').desc())
              .limit(1))

        rows=lp.collect()

        if not rows:
             return None   
        return rows[0]['last_processed_ingested_at']


3rd helper function 
update silver control table after successfull silver load

In [0]:
def upsert_silver_control(
    entity_name,
    last_processed_bronze_run_id,
    last_processed_ingested_at,
    rows_merged
):
    control_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                str(last_processed_bronze_run_id),
                last_processed_ingested_at,
                int(rows_merged),
                str(silver_run_id),
                "success",
                datetime.utcnow()
            )
        ],
        schema="""
            layer STRING,
            entity_name STRING,
            last_processed_bronze_run_id STRING,
            last_processed_ingested_at TIMESTAMP,
            rows_merged INT,
            silver_run_id STRING,
            run_status STRING,
            updated_at TIMESTAMP
        """
    )

    target_table = DeltaTable.forName(
        spark,
        "ecommerce_data.silver.silver_control"
    )

    (
        target_table.alias("target")
        .merge(
            control_df.alias("source"),
            """
            target.entity_name = source.entity_name
            AND target.layer = source.layer
            """
        )
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id":
                    "source.last_processed_bronze_run_id",
                "last_processed_ingested_at":
                    "source.last_processed_ingested_at",
                "rows_merged": "source.rows_merged",
                "silver_run_id": "source.silver_run_id",
                "run_status": "source.run_status",
                "updated_at": "source.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

4th helper function
get new data from bronze

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_updated_at=get_last_processed_bronze_ingested_at(entity_name)
    bronze_df=spark.read.table(bronze_table)
    if last_updated_at is None:
        return bronze_df,last_updated_at
    return bronze_df.filter(f.col('bronze_ingested_at')>f.lit(last_updated_at)),last_updated_at